# 1. Preparación de datos

In [ ]:
# Importe de librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.inspection import permutation_importance
from sklearn.tree import plot_tree

## Carga de Datos

In [ ]:
url = "https://raw.githubusercontent.com/Octaviochavez/analitica_clientes/main/data/dataset_clientes.csv"

In [ ]:
# Lectura de datos sucios del dataset desde directorio local data
data = pd.read_csv(url)
data.head()

## 1.1 Limpieza y Transformación de Datos

### Clases y Funciones necesarias

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
  """
  Eliminación de variables correlacionadas

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_to_drop_ : array-like
    Nombres de las columnas a eliminar.
  threshold : float
    Umbral de correlación.
  Returns
  -------
  DataFrame
    Conjunto de datos sin variables correlacionadas.
  """
  def __init__(self, threshold=0.9):
    self.threshold = threshold
    self.columns_to_drop_ = None

  def fit(self, X, y=None):
    X_df = pd.DataFrame(X)

    corr_matrix = X_df.corr().abs()
    upper = corr_matrix.where(
      np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    self.columns_to_drop_ = [
        col for col in upper.columns if any(upper[col] > self.threshold)
    ]

    return self

  def transform(self, X):
    X_df = pd.DataFrame(X)
    X_filtered = X_df.drop(columns=self.columns_to_drop_, errors="ignore")
    return X_filtered.values

In [ ]:
def evaluar(modelo: BaseEstimator, X_train: np.array, X_test: np.array, y_train: np.array, y_test: np.array):
  """
  Retorna las métricas del modelo

  Parámetros
  ----------
  modelo : BaseEstimator
    Modelo a evaluar.
  X_train : np.array
    Conjunto de datos de entrenamiento.
  X_test : np.array
    Conjunto de datos de prueba.
  y_train : np.array
    Etiquetas de entrenamiento.
  y_test : np.array
    Etiquetas de prueba
  Returns
  -------
  dict
    Diccionario con las métricas del modelo.

  """
  modelo.fit(X_train, y_train)

  y_pred = modelo.predict(X_test)
  y_prob = modelo.predict_proba(X_test)[:,1]

  return {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
  }

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
  """
  Ingeniería de características

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  Returns
  -------
  DataFrame
    Conjunto de datos con nuevas características.
  """
  def __init__(self):
    pass

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X = X.copy()
    
    # razón de endeudamiento
    X["ratio_endeudamiento"] = X["deuda_total"] / X["ingreso_mensual"]

    # porcentaje de gasto respecto al ingreso
    X['porcentaje_gasto'] = X['gasto_mensual'] / X['ingreso_mensual']
    
    return X

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        # Calculamos y guardamos los límites matemáticos AQUÍ (Fase de aprendizaje)
        self.lower_bounds_ = X_df.quantile(self.limits[0])
        self.upper_bounds_ = X_df.quantile(1 - self.limits[1])
        self.columns_ = X_df.columns if isinstance(X, pd.DataFrame) else np.arange(X.shape[1])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.columns_).copy()
        X_df = X_df.astype("float64")
        # Aplicamos los límites guardados
        for col in self.columns_:
            X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        return X_df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        else:
            return np.array(input_features)

In [ ]:
# Define función para eliminar duplicados

def tratar_duplicados(X : pd.DataFrame, drop = True):
  """
  Tratamiento de duplicados

  Parámetros
  ----------
  X : DataFrame
    Conjunto de datos.
  drop : bool
    Si se deben eliminar los duplicados.

  Retorna
  -------
  DataFrame
    Conjunto de datos sin duplicados.
  """
  return X.drop_duplicates() if drop else X

In [ ]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
  """
  Convierte un array en un DataFrame

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  feature_names_ : array-like
    Nombres de las columnas.
  Returns
  -------
  DataFrame
    Conjunto de datos con nombres de columnas.
  """
  def __init__(self, preprocessor):
    self.preprocessor = preprocessor
    self.feature_names_ = None

  def fit(self, X, y=None):
    # Obtener nombres después de fit del preprocessor
    self.feature_names_ = self.preprocessor.get_feature_names_out()
    return self

  def transform(self, X):
    return pd.DataFrame(X, columns=self.feature_names_)

### Detección de inconsistencias

In [ ]:
# Deteccion de inconsistencias: negativos o ceros en columnas que deberian ser positivas
columnas_negativas_check = [
    "ingreso_mensual", "gasto_mensual", "deuda_total", "score_crediticio",
    "edad", "antiguedad_meses", "frecuencia_compra", "ultima_compra_dias",
    "num_productos", "hora_registro",
]
columnas_ceros_check = [
    "ingreso_mensual", "gasto_mensual", "deuda_total", "score_crediticio",
    "edad", "antiguedad_meses",
]

conteo_negativos = {}
conteo_ceros = {}
for col in columnas_negativas_check:
    if col in data.columns:
        conteo_negativos[col] = int((data[col] < 0).sum())
for col in columnas_ceros_check:
    if col in data.columns:
        conteo_ceros[col] = int((data[col] == 0).sum())

conteo_negativos = {k: v for k, v in conteo_negativos.items() if v > 0}
conteo_ceros = {k: v for k, v in conteo_ceros.items() if v > 0}

print("Negativos detectados:", conteo_negativos if conteo_negativos else "ninguno")
print("Ceros detectados:", conteo_ceros if conteo_ceros else "ninguno")

In [ ]:
# Tratamiento: corrige valores negativos convirtiendolos a valor absoluto
columns_to_correct = [
    "ingreso_mensual", "gasto_mensual", "deuda_total", "score_crediticio",
    "edad", "antiguedad_meses", "frecuencia_compra", "ultima_compra_dias",
    "num_productos", "hora_registro",
]

for col in columns_to_correct:
    if col in data.columns:
        data[col] = data[col].abs()

print("Valores negativos corregidos a valores absolutos en las columnas relevantes.")

### Construcción de arrays para procesamiento

In [ ]:
numerical_features = ["ratio_endeudamiento", "porcentaje_gasto", "gasto_mensual", "score_crediticio", "ingreso_mensual", "deuda_total", "edad", "antiguedad_meses", "frecuencia_compra", "ultima_compra_dias", "num_productos", "hora_registro"] # Define listado de variables numéricas
categorical_nominales = ["tiene_tarjeta_credito", "genero", "region", "estado_civil", "canal_registro","dia_semana_registro"] # Define listado de variables categóricas nominales
categorical_ordinales = ["tipo_plan", "uso_app"] # Define listado de variables categóricas ordinales
date_time_features = ["fecha_registro"] # Define listado de variables de fecha y hora
orden_tipo_plan = ['Basico', 'Estandar', 'Premium'] # Define orden para variable ordinal tipo_plan
orden_uso_app = ['Bajo', 'Medio', 'Alto'] # Define orden para variable ordinal uso_app

### Pipelines para cada tipo de conjunto


In [ ]:
# Define pipeline para variables numéricas
pipeline_numerical_features = Pipeline(steps=[
    ('winsorizer', Winsorizer(limits=(0.05, 0.05))), # Aplica Winsorización para limitar outliers al 5%
    ('imputer', SimpleImputer(strategy='mean')), # Imputa valores faltantes con el promedio
    ('scaler', StandardScaler()) # Escala características numéricas
])

In [ ]:
# Define pipeline para variables categóricas nominales
pipeline_nominales = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputa valores faltantes con la moda
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Aplica codificación OneHotEncoder para variables nominales
])

In [ ]:
# Define pipeline para variables categóricas ordinales
pipeline_ordinales = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputa valores faltantes con la moda
    ('ordinal', OrdinalEncoder(categories=[orden_tipo_plan, orden_uso_app])) # Aplica codificación OrdinalEncoder para variables ordinales con orden definido
])

### Integración de pipelines de transformación


In [ ]:
# Combina pipelines para aplicar transformaciones específicas a cada tipo de variable
preprocesador = ColumnTransformer(
    transformers=[
        ('num_limpios', pipeline_numerical_features, numerical_features),
        ('cat_nom', pipeline_nominales, categorical_nominales),
        ('cat_ord', pipeline_ordinales, categorical_ordinales),
    ],
    remainder='drop' # Elimina columnas no especificadas
)

### Elimina duplicados y separa objetivo/features para mantener alineacion

In [ ]:
data_sin_dup = tratar_duplicados(data, drop=True).reset_index(drop=True)
var_dep = data_sin_dup["abandono"].reset_index(drop=True)
var_indep = data_sin_dup.drop(columns=["abandono"])

### Pipeline para Regresión Logística


In [ ]:
# Define pipeline de Regresión Logística que incluye eliminación de duplicados, ingeniería de características y preprocesamiento específico para cada tipo de variable
pipeline_modelo_lr = Pipeline(
    steps=[
        ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": True})), # Elimina duplicados del dataset
        ("feature_engineering", FeatureEngineering()), # Agrega nuevas características
        ("preprocesamiento", preprocesador),
        ("conversion", DataFrameConverter(preprocesador)), # Convierte la salida a DataFrame con nombres
        ("colinealidad", CorrelationFilter(threshold = 0.9)), # Elimina variables altamente correlacionadas
        ("modelo", LogisticRegression(max_iter = 1000)) # Agrega modelo de regresión logística al pipeline
    ]
)

# 2. Modelamiento

## 2.1 Entendimiento de los datos

In [ ]:
target = 'abandono'

In [ ]:
# Revisa la distribución de la variable objetivo
# En este caso se obtiene un gráfico de torta
data[target].value_counts().plot(kind='pie', autopct='%1.1f%%',
                                 labels=['No abandona', 'Abandona'],
                                 figsize=(6, 6))
plt.title("Distribución de variable objetivo", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Grafica la distribución de la variable objetivo
data[target].value_counts().plot(kind="bar", color=["red", "green"])
plt.title("Distribución de la variable objetivo", fontweight="bold", fontsize=14)
plt.xlabel("Clase", fontweight="bold", fontsize=12)
plt.ylabel("Frecuencia", fontweight="bold", fontsize=12)
# Agrega los nombres de las clases en el eje (0 = No, 1 = Si)
plt.xticks([0, 1], ["No abandona", "Abandona"])
# Agrega las etiquetas a cada clase
for i, count in enumerate(data[target].value_counts()):
    plt.text(i, count, str(count), ha='center', va='bottom')
plt.show()

## 2.2 Modelo de Regresión Logística

### Variables

In [ ]:
url = "https://raw.githubusercontent.com/Octaviochavez/analitica_clientes/rama_david/data/dataset_clientes_limpio.csv"
data = pd.read_csv(url)

In [ ]:
# X = data.drop(columns=["abandono"])
# y = data[target]

### División train/test


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=29, stratify = y)

### Evaluación


In [ ]:
metricas_lr = evaluar(pipeline_modelo_lr, X_train, X_test, y_train, y_test)
print(f"{'Accuracy':<20}: {metricas_lr['accuracy']:.4f}")
print(f"{'F1 Score:':<20}: {metricas_lr['f1']:.4f}")
print(f"{'ROC AUC Score':<20}: {metricas_lr['roc_auc']:.4f}")